# Simulación del problema bancario con tres cajas independientes

## Introducción

Este notebook resuelve la actividad de teoría de colas del problema bancario. El banco opera durante **480 minutos**, recibe clientes que realizan **retiros (70 %)** o **pagos/consignaciones (30 %)** y dispone inicialmente de tres cajas.

El objetivo es ejecutar múltiples réplicas, medir el desempeño de cada caja y comparar la configuración actual con dos distribuciones exclusivas: **una caja para retiros y dos para pagos**, y **dos cajas para retiros y una para pagos**. También se compara una alternativa de cuatro cajas para evaluar si hace falta ampliar la capacidad.

> **Decisión de modelado:** se implementan tres colas FIFO independientes, coherentes con la indicación del enunciado de tratar las cajas como M/M/1. Cuando varias cajas pueden atender una acción, el cliente elige la que tiene menor carga pendiente. Los clientes dejan de llegar al minuto 480, pero quienes ya llegaron terminan su atención.


## Metodología y supuestos

1. Cada cliente recibe una acción (`retiro` o `pago`) y un tipo (`rapido`, `normal`, `lento`, `muy_lento`) según las probabilidades del enunciado.
2. El tiempo hasta la siguiente llegada y el tiempo de servicio se generan con distribuciones exponenciales cuyas medias dependen de la acción y del tipo.
3. Cada caja conserva su propia cola FIFO. Entre las cajas elegibles se selecciona la de menor carga pendiente; los empates se resuelven aleatoriamente para evitar favorecer siempre la primera caja.
4. Se ejecutan **30 réplicas por escenario** con semillas independientes y reproducibles.
5. Para decidir si se necesita una cuarta caja se usa un criterio explícito de servicio: espera media general mayor que 5 minutos o percentil 95 mayor que 10 minutos. Estos umbrales son supuestos de análisis y pueden modificarse.
6. La mezcla de tipos produce un sistema más general que un M/M/1 homogéneo. Por eso las fórmulas clásicas se usan como referencia conceptual y la validación principal se hace con invariantes, reproducibilidad y Ley de Little observada.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from math import sqrt
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HORIZONTE_MIN = 8 * 60
N_REPLICAS = 30
SEMILLA_BASE = 202503
UMBRAL_ESPERA_MEDIA = 5.0
UMBRAL_ESPERA_P95 = 10.0

ACCIONES = ("retiro", "pago")
TIPOS = ("rapido", "normal", "lento", "muy_lento")

PROB_ACCION = {"retiro": 0.70, "pago": 0.30}
PROB_TIPO = {
    "retiro": {"rapido": 0.23, "normal": 0.40, "lento": 0.17, "muy_lento": 0.20},
    "pago": {"rapido": 0.10, "normal": 0.20, "lento": 0.30, "muy_lento": 0.40},
}

# Medias (minutos) tomadas de la Tabla 1 del enunciado.
PARAMETROS = {
    "retiro": {
        "rapido": {"servicio": 1.0, "interllegada": 1.0},
        "normal": {"servicio": 2.0, "interllegada": 2.0},
        "lento": {"servicio": 3.0, "interllegada": 3.0},
        "muy_lento": {"servicio": 4.0, "interllegada": 3.0},
    },
    "pago": {
        "rapido": {"servicio": 3.0, "interllegada": 1.0},
        "normal": {"servicio": 3.0, "interllegada": 2.0},
        "lento": {"servicio": 5.0, "interllegada": 3.0},
        "muy_lento": {"servicio": 7.0, "interllegada": 4.0},
    },
}


In [ ]:
# Cada escenario define las acciones que puede atender cada caja.
ESCENARIOS = {
    "tres_mixtas": {
        "Caja 1": ("retiro", "pago"),
        "Caja 2": ("retiro", "pago"),
        "Caja 3": ("retiro", "pago"),
    },
    "una_retiros_dos_pagos": {
        "Caja 1": ("retiro",),
        "Caja 2": ("pago",),
        "Caja 3": ("pago",),
    },
    "dos_retiros_un_pago": {
        "Caja 1": ("retiro",),
        "Caja 2": ("retiro",),
        "Caja 3": ("pago",),
    },
    "cuatro_mixtas": {
        "Caja 1": ("retiro", "pago"),
        "Caja 2": ("retiro", "pago"),
        "Caja 3": ("retiro", "pago"),
        "Caja 4": ("retiro", "pago"),
    },
}


def validar_parametros(parametros: dict) -> None:
    '''Falla temprano si las probabilidades o medias no son válidas.'''
    if not np.isclose(sum(PROB_ACCION.values()), 1.0):
        raise ValueError("Las probabilidades de acción deben sumar 1.")

    for accion in ACCIONES:
        if not np.isclose(sum(PROB_TIPO[accion].values()), 1.0):
            raise ValueError(f"Las probabilidades de tipo para {accion} deben sumar 1.")
        for tipo in TIPOS:
            valores = parametros[accion][tipo]
            if valores["servicio"] <= 0 or valores["interllegada"] <= 0:
                raise ValueError("Todas las medias exponenciales deben ser positivas.")

    for nombre, roles in ESCENARIOS.items():
        cubiertas = {accion for acciones in roles.values() for accion in acciones}
        if cubiertas != set(ACCIONES):
            raise ValueError(f"El escenario {nombre} no cubre todas las acciones.")


validar_parametros(PARAMETROS)


## Motor de simulación

El motor genera una secuencia cronológica de clientes y actualiza la disponibilidad de cada caja. La carga pendiente de una caja en el momento de llegada es `max(disponible_en - llegada, 0)`. Esta cantidad resume el trabajo de los clientes que ya están en esa cola.

Se guardan los tiempos de llegada, inicio, fin, servicio, espera y permanencia en el sistema. Esto permite verificar el orden temporal y calcular indicadores sin depender de variables internas ocultas.


In [ ]:
@dataclass(frozen=True)
class ClienteGenerado:
    id_cliente: int
    llegada: float
    accion: str
    tipo: str
    tiempo_servicio: float


def _elegir(rng: np.random.Generator, probabilidades: dict[str, float]) -> str:
    opciones = list(probabilidades)
    pesos = np.array([probabilidades[opcion] for opcion in opciones], dtype=float)
    return str(rng.choice(opciones, p=pesos))


def generar_clientes(
    rng: np.random.Generator,
    horizonte: float = HORIZONTE_MIN,
    parametros: dict = PARAMETROS,
) -> list[ClienteGenerado]:
    '''Genera llegadas hasta el cierre usando las medias de la actividad.'''
    clientes: list[ClienteGenerado] = []
    reloj = 0.0
    siguiente_id = 1

    while True:
        accion = _elegir(rng, PROB_ACCION)
        tipo = _elegir(rng, PROB_TIPO[accion])
        media_llegada = parametros[accion][tipo]["interllegada"]
        reloj += float(rng.exponential(media_llegada))
        if reloj > horizonte:
            break

        media_servicio = parametros[accion][tipo]["servicio"]
        clientes.append(
            ClienteGenerado(
                id_cliente=siguiente_id,
                llegada=reloj,
                accion=accion,
                tipo=tipo,
                tiempo_servicio=float(rng.exponential(media_servicio)),
            )
        )
        siguiente_id += 1

    return clientes


def _caja_menos_cargada(
    rng: np.random.Generator,
    accion: str,
    llegada: float,
    disponible_en: dict[str, float],
    roles: dict[str, tuple[str, ...]],
) -> str:
    elegibles = [caja for caja, acciones in roles.items() if accion in acciones]
    if not elegibles:
        raise RuntimeError(f"No existe una caja elegible para {accion}.")

    cargas = {caja: max(disponible_en[caja] - llegada, 0.0) for caja in elegibles}
    carga_minima = min(cargas.values())
    empatadas = [caja for caja, carga in cargas.items() if np.isclose(carga, carga_minima)]
    return str(rng.choice(empatadas))


def simular_replica(
    escenario: str,
    semilla: int,
    horizonte: float = HORIZONTE_MIN,
    parametros: dict = PARAMETROS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Simula una jornada y devuelve detalle de clientes y resumen por caja.'''
    validar_parametros(parametros)
    if escenario not in ESCENARIOS:
        raise KeyError(f"Escenario desconocido: {escenario}")

    rng = np.random.default_rng(semilla)
    roles = ESCENARIOS[escenario]
    disponible_en = {caja: 0.0 for caja in roles}
    registros: list[dict] = []

    for cliente in generar_clientes(rng, horizonte, parametros):
        caja = _caja_menos_cargada(
            rng, cliente.accion, cliente.llegada, disponible_en, roles
        )
        inicio = max(cliente.llegada, disponible_en[caja])
        fin = inicio + cliente.tiempo_servicio
        disponible_en[caja] = fin

        registros.append(
            {
                "id_cliente": cliente.id_cliente,
                "escenario": escenario,
                "cajero": caja,
                "accion": cliente.accion,
                "tipo": cliente.tipo,
                "llegada": cliente.llegada,
                "inicio_servicio": inicio,
                "fin_servicio": fin,
                "servicio": cliente.tiempo_servicio,
                "espera": inicio - cliente.llegada,
                "tiempo_sistema": fin - cliente.llegada,
            }
        )

    clientes = pd.DataFrame(registros)
    resumen_filas = []
    for caja in roles:
        grupo = clientes.loc[clientes["cajero"] == caja]
        ocupado_hasta_cierre = sum(
            max(0.0, min(fin, horizonte) - min(max(inicio, 0.0), horizonte))
            for inicio, fin in zip(grupo["inicio_servicio"], grupo["fin_servicio"])
            if inicio < horizonte
        )
        resumen_filas.append(
            {
                "escenario": escenario,
                "cajero": caja,
                "acciones": "/".join(roles[caja]),
                "usuarios_atendidos": len(grupo),
                "atencion_promedio": grupo["servicio"].mean(),
                "espera_promedio": grupo["espera"].mean(),
                "espera_p95": grupo["espera"].quantile(0.95),
                "tiempo_sistema_promedio": grupo["tiempo_sistema"].mean(),
                "utilizacion_hasta_cierre": ocupado_hasta_cierre / horizonte,
                "fin_ultimo_servicio": grupo["fin_servicio"].max(),
            }
        )

    resumen = pd.DataFrame(resumen_filas).fillna(0.0)
    return clientes, resumen


In [ ]:
def ejecutar_replicas(
    escenario: str,
    n_replicas: int = N_REPLICAS,
    semilla_base: int = SEMILLA_BASE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Ejecuta réplicas con semillas independientes derivadas de SeedSequence.'''
    if n_replicas < 10:
        raise ValueError("La actividad exige ejecutar al menos 10 réplicas.")

    semillas = np.random.SeedSequence(semilla_base).spawn(n_replicas)
    detalles = []
    resumenes = []

    for replica, secuencia in enumerate(semillas, start=1):
        semilla = int(secuencia.generate_state(1, dtype=np.uint32)[0])
        clientes, resumen = simular_replica(escenario, semilla)
        clientes.insert(0, "replica", replica)
        resumen.insert(0, "replica", replica)
        detalles.append(clientes)
        resumenes.append(resumen)

    return pd.concat(detalles, ignore_index=True), pd.concat(resumenes, ignore_index=True)


def intervalo_confianza_95(valores: Iterable[float]) -> pd.Series:
    '''Calcula media e IC95 % normal sin depender de SciPy.'''
    serie = pd.Series(list(valores), dtype=float).dropna()
    media = serie.mean()
    error = 1.96 * serie.std(ddof=1) / sqrt(len(serie))
    return pd.Series({"media": media, "ic95_inferior": media - error, "ic95_superior": media + error})


def validar_replica(clientes: pd.DataFrame, resumen: pd.DataFrame, escenario: str) -> None:
    '''Comprueba conservación, tiempos y elegibilidad de las asignaciones.'''
    if len(clientes) != int(resumen["usuarios_atendidos"].sum()):
        raise AssertionError("La conservación de clientes no se cumple.")
    if (clientes["espera"] < -1e-12).any():
        raise AssertionError("Se detectaron tiempos de espera negativos.")
    if (clientes["inicio_servicio"] < clientes["llegada"] - 1e-12).any():
        raise AssertionError("Un servicio inicia antes de la llegada.")
    if (clientes["fin_servicio"] < clientes["inicio_servicio"] - 1e-12).any():
        raise AssertionError("Un servicio termina antes de iniciar.")

    roles = ESCENARIOS[escenario]
    for fila in clientes.itertuples(index=False):
        if fila.accion not in roles[fila.cajero]:
            raise AssertionError("Un cliente fue enviado a una caja no elegible.")


def validar_little(clientes: pd.DataFrame) -> pd.DataFrame:
    '''Verifica Lq = lambda * Wq por caja usando áreas observadas.'''
    filas = []
    for (replica, escenario, cajero), grupo in clientes.groupby(["replica", "escenario", "cajero"]):
        horizonte_observado = max(float(grupo["fin_servicio"].max()), HORIZONTE_MIN)
        tasa = len(grupo) / horizonte_observado
        wq = float(grupo["espera"].mean())
        lq_por_area = float(grupo["espera"].sum()) / horizonte_observado
        filas.append(
            {
                "replica": replica,
                "escenario": escenario,
                "cajero": cajero,
                "Lq_area": lq_por_area,
                "lambda_Wq": tasa * wq,
                "error_absoluto": abs(lq_por_area - tasa * wq),
            }
        )
    return pd.DataFrame(filas)


## Resultados, validación y conclusiones

Las siguientes celdas se completan con la ejecución de las réplicas, las tablas solicitadas, las validaciones y las recomendaciones finales.
